In [9]:
import torch
import torch.nn as nn
from torchvision import transforms
from PIL import Image
from pathlib import Path
import numpy as np


# ============================================================
# 0. Paths and settings
# ============================================================

REPO_DIR = Path(r"C:\Users\Wang2_h\dinov2")

WEIGHTS_NAME = "cell_dino_vits8_pretrain_cp-37d20e9c.pth"
CELL_DINO_WEIGHTS = REPO_DIR / "checkpoints" / WEIGHTS_NAME

CLASSIFIER_WEIGHTS = REPO_DIR / "best_celldino_symbiont_classifier_cp_vits8.pth"

IMAGE_PATH = REPO_DIR / "test_images" / "H2_20x_5_C2_red_crop_4.png"

MODEL_NAME = "cell_dino_cp_vits8"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"


# ============================================================
# 1. Load classifier checkpoint
# ============================================================

checkpoint = torch.load(CLASSIFIER_WEIGHTS, map_location=DEVICE)

print("Checkpoint keys:")
print(checkpoint.keys())

print("\nSaved head keys:")
for k, v in checkpoint["head_state_dict"].items():
    print(k, v.shape)


# Use settings saved during training
class_to_idx = checkpoint["class_to_idx"]
idx_to_class = {v: k for k, v in class_to_idx.items()}

IMAGE_SIZE = checkpoint.get("image_size", 224)
INPUT_DIM = checkpoint.get("input_dim", 384)
NUM_CLASSES = checkpoint.get("num_classes", 2)

print("\nClass mapping:")
print(class_to_idx)

print("\nIMAGE_SIZE:", IMAGE_SIZE)
print("INPUT_DIM:", INPUT_DIM)
print("NUM_CLASSES:", NUM_CLASSES)
print("DEVICE:", DEVICE)


# ============================================================
# 2. Same preprocessing as validation during training
# ============================================================

class GrayscaleTo5Channel:
    def __call__(self, img):
        """
        Convert PIL image to 5-channel tensor for CellDINO CP model.

        Input:
            PIL image

        Output:
            Tensor [5, H, W], values in [0, 1]
        """
        img = img.convert("L")
        arr = np.asarray(img).astype(np.float32) / 255.0
        arr5 = np.stack([arr, arr, arr, arr, arr], axis=0)

        return torch.from_numpy(arr5)


val_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),

    GrayscaleTo5Channel(),

    transforms.Normalize(
        mean=[0.5, 0.5, 0.5, 0.5, 0.5],
        std=[0.5, 0.5, 0.5, 0.5, 0.5],
    )
])


# ============================================================
# 3. Classification head
# Must match your training code exactly
# ============================================================

class CellDINOClassificationHead(nn.Module):
    def __init__(self, input_dim=384, num_classes=2, dropout=0.2):
        super().__init__()

        self.classifier = nn.Sequential(
            nn.Linear(input_dim, 512),
            nn.LayerNorm(512),
            nn.GELU(),
            nn.Dropout(dropout),

            nn.Linear(512, 256),
            nn.LayerNorm(256),
            nn.GELU(),
            nn.Dropout(dropout),

            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        """
        x: [batch_size, 384]
        return: [batch_size, num_classes]
        """
        return self.classifier(x)


# ============================================================
# 4. CellDINO classifier wrapper
# Must match your training code
# ============================================================

class CellDINOClassifier(nn.Module):
    def __init__(
        self,
        celldino_backbone,
        input_dim=384,
        num_classes=2,
        freeze_backbone=True
    ):
        super().__init__()

        self.backbone = celldino_backbone

        self.head = CellDINOClassificationHead(
            input_dim=input_dim,
            num_classes=num_classes
        )

        if freeze_backbone:
            for p in self.backbone.parameters():
                p.requires_grad = False

    def extract_features(self, images):
        """
        images: [batch_size, 5, H, W]
        features: [batch_size, 384]
        """

        if hasattr(self.backbone, "forward_features"):
            output = self.backbone.forward_features(images)
        else:
            output = self.backbone(images)

        if isinstance(output, dict):
            if "x_norm_clstoken" in output:
                features = output["x_norm_clstoken"]
            elif "cls_token" in output:
                features = output["cls_token"]
            elif "global_embedding" in output:
                features = output["global_embedding"]
            else:
                raise KeyError(
                    f"Cannot find global feature in CellDINO output. "
                    f"Available keys: {output.keys()}"
                )

        elif isinstance(output, torch.Tensor):
            features = output

        elif isinstance(output, (tuple, list)):
            features = output[0]

        else:
            raise TypeError(f"Unsupported CellDINO output type: {type(output)}")

        if features.ndim != 2:
            raise ValueError(
                f"Expected features shape [batch_size, feature_dim], "
                f"but got {features.shape}"
            )

        return features

    def forward(self, images):
        if not any(p.requires_grad for p in self.backbone.parameters()):
            with torch.no_grad():
                features = self.extract_features(images)
        else:
            features = self.extract_features(images)

        logits = self.head(features)

        return logits


# ============================================================
# 5. Load CellDINO backbone
# ============================================================

def load_celldino_backbone(repo_dir, model_name, weights_path, device):
    backbone = torch.hub.load(
        str(repo_dir),
        model_name,
        source="local",
        pretrained_path=str(weights_path),
    )

    backbone = backbone.to(device)
    backbone.eval()

    return backbone


celldino_backbone = load_celldino_backbone(
    repo_dir=REPO_DIR,
    model_name=MODEL_NAME,
    weights_path=CELL_DINO_WEIGHTS,
    device=DEVICE
)


# ============================================================
# 6. Build model and load trained head
# ============================================================

model = CellDINOClassifier(
    celldino_backbone=celldino_backbone,
    input_dim=INPUT_DIM,
    num_classes=NUM_CLASSES,
    freeze_backbone=True
)

model = model.to(DEVICE)

# Important:
# Your training saved model.head.state_dict(),
# so we load into model.head directly.
model.head.load_state_dict(checkpoint["head_state_dict"])

model.eval()

print("\nClassifier head loaded successfully.")


# ============================================================
# 7. Run single-image classification
# ============================================================

img = Image.open(IMAGE_PATH)

x = val_transform(img)
x = x.unsqueeze(0)       # [5, H, W] -> [1, 5, H, W]
x = x.to(DEVICE)

with torch.no_grad():
    logits = model(x)
    probs = torch.softmax(logits, dim=1)

pred_idx = torch.argmax(probs, dim=1).item()
pred_class = idx_to_class[pred_idx]
confidence = probs[0, pred_idx].item()


# ============================================================
# 8. Print result
# ============================================================

print("\nPrediction result")
print("=================")
print("Image:", IMAGE_PATH)
print("Logits:", logits.cpu().numpy())
print("Probabilities:", probs.cpu().numpy())
print("Predicted index:", pred_idx)
print("Predicted class:", pred_class)
print("Confidence:", confidence)

print("\nAll class probabilities:")
for idx, prob in enumerate(probs[0]):
    class_name = idx_to_class[idx]
    print(f"{class_name}: {prob.item():.4f}")

C:\Users\Wang2_h\AppData\Local\Temp\ipykernel_27624\1692917692.py:30: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(CLASSIFIER_WEIGHTS, map_location=

Checkpoint keys:
dict_keys(['head_state_dict', 'class_to_idx', 'image_size', 'input_dim', 'num_classes', 'model_name', 'weights_name'])

Saved head keys:
classifier.0.weight torch.Size([512, 384])
classifier.0.bias torch.Size([512])
classifier.1.weight torch.Size([512])
classifier.1.bias torch.Size([512])
classifier.4.weight torch.Size([256, 512])
classifier.4.bias torch.Size([256])
classifier.5.weight torch.Size([256])
classifier.5.bias torch.Size([256])
classifier.8.weight torch.Size([2, 256])
classifier.8.bias torch.Size([2])

Class mapping:
{'symbiont_A': 0, 'symbiont_B': 1}

IMAGE_SIZE: 224
INPUT_DIM: 384
NUM_CLASSES: 2
DEVICE: cuda

Classifier head loaded successfully.

Prediction result
Image: C:\Users\Wang2_h\dinov2\test_images\H2_20x_5_C2_red_crop_4.png
Logits: [[-0.11198308 -0.09626085]]
Probabilities: [[0.4960695  0.50393045]]
Predicted index: 1
Predicted class: symbiont_B
Confidence: 0.5039304494857788

All class probabilities:
symbiont_A: 0.4961
symbiont_B: 0.5039


In [4]:
checkpoint = torch.load(CLASSIFIER_WEIGHTS, map_location=DEVICE)

print(checkpoint.keys())

print("\nSaved head keys:")
for k, v in checkpoint["head_state_dict"].items():
    print(k, v.shape)

dict_keys(['head_state_dict', 'class_to_idx', 'image_size', 'input_dim', 'num_classes', 'model_name', 'weights_name'])

Saved head keys:
classifier.0.weight torch.Size([512, 384])
classifier.0.bias torch.Size([512])
classifier.1.weight torch.Size([512])
classifier.1.bias torch.Size([512])
classifier.4.weight torch.Size([256, 512])
classifier.4.bias torch.Size([256])
classifier.5.weight torch.Size([256])
classifier.5.bias torch.Size([256])
classifier.8.weight torch.Size([2, 256])
classifier.8.bias torch.Size([2])


C:\Users\Wang2_h\AppData\Local\Temp\ipykernel_27624\3599780848.py:1: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(CLASSIFIER_WEIGHTS, map_location=D